In [0]:
%sql
CREATE CATALOG retail_catalog
MANAGED LOCATION 'abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Bronze';

In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
USE CATALOG retail_catalog;

In [0]:
spark

## Bronze Layer

In [0]:
df_transactions = spark.read.parquet('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Bronze/Transactions/*')
df_products = spark.read.parquet('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Bronze/Products/*')
df_stores = spark.read.parquet('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Bronze/Stores/*')
df_customers = spark.read.parquet('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Bronze/Customers/customer_https/azure-data-engineer---multi-source/refs/heads/main/customers.parquet')

### Silver Layer - Data Cleaning

In [0]:
from pyspark.sql.functions import col

# Convert types and clean data
df_transactions = df_transactions.select(
    col("transaction_id").cast("integer"),
    col("customer_id").cast("integer"),
    col("product_id").cast("integer"),
    col("store_id").cast("integer"),
    col("quantity").cast("integer"),
    col("transaction_date").cast("date")
)

df_products = df_products.select(
    col("product_id").cast("integer"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

df_stores = df_stores.select(
    col("store_id").cast("integer"),
    col("store_name"),
    col("location")
)

df_customers = df_customers.select(
    "customer_id","first_name","last_name","email","phone","city","registration_date"
).dropDuplicates(["customer_id"])

In [0]:
# Join all the data
from pyspark.sql.functions import col

df_silver = df_transactions \
    .join(df_customers, "customer_id") \
    .join(df_products, "product_id") \
    .join(df_stores, "store_id") \
    .withColumn("total_amount", col("quantity") * col("price"))

In [0]:
display(df_silver)

In [0]:
df_silver.write.mode("overwrite").format("delta").save('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Silver/Cleaned_retail')

In [0]:
%sql

CREATE TABLE retail_catalog.default.retail_silver_cleaned
USING DELTA
LOCATION 'abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Silver/Cleaned_retail'

In [0]:
%sql
SELECT * FROM retail_catalog.default.retail_silver_cleaned;

### Gold Layer

In [0]:
silver_df = spark.read.format("delta").load('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Silver/Cleaned_retail')

In [0]:
from pyspark.sql.functions import sum, countDistinct, avg

gold_df = silver_df.groupBy(
    "transaction_date",
    "product_id", "product_name", "category",
    "store_id", "store_name", "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)


In [0]:
display(gold_df)

In [0]:
gold_df.write.mode("overwrite").format("delta").save('abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Gold/Aggregated_data')

In [0]:
%sql
DROP TABLE IF EXISTS retail_catalog.default.retail_gold;
CREATE TABLE retail_catalog.default.retail_gold
USING DELTA
LOCATION 'abfss://retaildata@azureretalprojectsa.dfs.core.windows.net/Gold/Aggregated_data';

In [0]:
%sql
SELECT * FROM retail_gold;